# Exploring Uncertainty in Climate Data

There are several kinds of scientific uncertainty that arise when working with long-term projections of future climates:
1. Model Uncertainty, which illustrates the differences between different models (namely, how model physics, settings, or parameters can change the outcome)
2. Scenario Uncertainty, which illustrates the differences in outcomes between end-of-century emissions targets
3. Internal Variability, which represents the variations inherent within the climate system itself

This notebook begins to explore **Model Uncertainty** in the Cal-Adapt: Analytics Engine by focusing on **temperature trends** across downscaled simulations. We also compare the suite of models currently available in the [Cal-Adapt Data Catalog](https://analytics.cal-adapt.org/data/) to the full set of CMIP6 models to illustrate the diffrences between our models and all available models

---
**General Outline So Far**
1. Read in & display CAE models' temperature trend range
2. Read in & display CMIP6 archive temperature trend range, with CAE models highlighted
3. Display postage stamps of CAE models + selection (all?) CMIP6 models statistics at selected warming level
---

# Development notes:
- think about warming levels framework
- provide postage stamps
- statistics of comparisons
- want to understand warming level range due to cross-model differences, highlight change at end of century
- want a CA focus, with maps and summary statistics for California
- compare warming levels and CMIP6 archive

**Important note to self** Working with the @add_derived_variables branch (branched off of catalog_restructure)

# Step 0: Setup
Import useful packages and libraries

In [ ]:
import intake
import pandas as pd
import hvplot.xarray # noqa
import hvplot.pandas
import xarray as xr
import holoviews as hv

import dask

# from cmip6_preprocessing.preprocessing import combined_preprocessing

import panel as pn
pn.extension()

import climakitae as ck

# Step 1: Load in CAE Models
Load in Cal-Adapt: Analytics Engine mdoels

In [ ]:
app = ck.Application()

In [ ]:
# selecting monthly 1980-2100, SSP3-7.0 with historical climate appended, air temperature in degC, at 45 km, area averaged, for California
app.select()

In [ ]:
hist_ssp370 = app.retrieve().isel(scenario=0).compute()

In [ ]:
# Grabbing a historical baseline, 1981-2010 for now to start illustrative process
app.selections.time_slice = (1981,2010)
app.scenario = "Historical"
historical_baseline = app.retrieve().isel(scenario=0).compute()
historical_baseline_annual = historical_baseline.groupby("time.year").mean(dim="time") # smoothing to make annual timeseries easier to read

To do: annual smoothing to make plots more easily readable

In [ ]:
h = historical_baseline_annual.hvplot.line(x='year',
                    by='simulation',
                    title='Historical Baseline')

h

To do: warming levels plot connection? difference from 1850-2010 baseline?

In [ ]:
# Using 1980-2015 historical data as an example at first
historical_baseline_annual_vals = historical_baseline_annual.mean(dim="year") # takes the long-term average over the entire historical period per simulation
ae_warmlevel = hist_ssp370_annual - historical_baseline_annual_vals

In [ ]:
warm_level = 3.0
warmlevel_line = hv.HLine(warm_level).opts(color="black", line_width=1.0) \
                * hv.Text(x=1993, y=warm_level+0.25, text=".   " + str(warm_level) + "°C warming level").opts(style=dict(text_font_size='8pt'))

ae_timseries = ae_warmlevel.hvplot.line(x='year',
                    by='simulation',
                    title='When do the CAE models reach the warming level? \nRegion: {}'.format(app.selections.location.cached_area))

ae_timseries * warmlevel_line

**Idea**: Like in warming_levels, it would be great to show/extract what year(s) the warming_level is crossed, for postage stamp views in comparison to CMIP6

# Step 2: Load data in for the full CMIP6 archive

In [ ]:
# Raw catalog csv
df = pd.read_csv("https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.csv")
df['experiment_id'].unique()

In [ ]:
## Does difference in grid label (native vs. gr vs. gr1) matter for our purposes?

# CMIP6 catalog
col = intake.open_esm_datastore('https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.json')
cat = col.search(
    table_id = "Amon",
    variable_id = "tas", # surface air temperature
    experiment_id = ["historical"],
    activity_id = "CMIP",
    member_id = "r1i1p1f1"
    # source_id = "ACCESS-CM2" # specific model not in CAE
    # source_id = "CESM2" # specific model in CAE
)

dsets = cat.to_dataset_dict(zarr_kwargs={'consolidated': True}, storage_options={"anon": True})
ds_key = list(dsets.keys())

# ds_key 
## 55 models with r1i1p1f1 in historical

In [ ]:
ds = dsets['CMIP.MPI-M.MPI-ESM1-2-HR.historical.Amon.gn'] # single model not in cae
ds

In [ ]:
ds2 = dsets['CMIP.NOAA-GFDL.GFDL-ESM4.historical.Amon.gr1']
ds3 = dsets['CMIP.MIROC.MIROC6.historical.Amon.gn']

In [ ]:
ds = ds - 273.15 # convert to degC
ds_ts = ds.groupby("time.year").mean(dim=["lat", "lon", "bnds"])
ds_ts_annual = ds_ts.groupby("time.year").mean(dim="time")
# ds.unit = "°C"

In [ ]:
ds2 = ds2 - 273.15 # convert to degC
ds2_ts = ds2.groupby("time.year").mean(dim=["lat", "lon", "bnds"])
ds2_ts_annual = ds2_ts.groupby("time.year").mean(dim="time")
# ds2.unit = "°C"

In [ ]:
ds3 = ds3 - 273.15 # convert to degC
ds3_ts = ds3.groupby("time.year").mean(dim=["lat", "lon", "bnds"])
ds3_ts_annual = ds3_ts.groupby("time.year").mean(dim="time")
# ds2.unit = "°C"

In [ ]:
## BIG CAVEAT: this is displaying CALIFORNIA models and comparing to GLOBAL -- to do next
## Check data resolution.... CA data is 45 km, global will vary per model depending on resolution

## Show full CMIP6 timeslice from 1850, or slice at 1950 onwards? 

c = ds_ts_annual.hvplot.line(x='year',
                            title="Historical comparison")

c2 = ds2_ts_annual.hvplot.line(x="year")

c3 = ds3_ts_annual.hvplot.line(x="year")

c * c2 * c3 * h

**To do**: Historical baseline for the CMIP6 archive, in order to do warming level differences, use the MMEM? 

**Idea**: Eventual goal is to have a nice figure that shows the historical + future timeline (1980-2100) 
- CMIP6 archive will be muted colors, individual models as thin lines
- CMIP6 MMEM will be a bolder line
- CAE models will be in identifiable colors

In [ ]:
# this is poor coding, just setting up example brute-force code here for testing of concept
mmem = (ds_ts_annual + ds2_ts_annual + ds3_ts_annual) / 3
d1_base = ds_ts_annual - mmem
d2_base = ds2_ts_annual - mmem
d3_base = ds3_ts_annual - mmem

In [ ]:
ind_color = 'grey'
lw = 0.75
c = ds_ts_annual.hvplot.line(x='year', line_width=lw, color=ind_color, xmin=1980)
c2 = ds2_ts_annual.hvplot.line(x="year", line_width=lw, color=ind_color)
c3 = ds3_ts_annual.hvplot.line(x="year", line_width=lw, color=ind_color)

m = mmem.hvplot.line(x="year", color='black', line_width=2)

c * c2 * c3 * m #* h

---
Useful to have a "When do the CMIP6 models/multi-model ensemble mean reach the warming level" for comparison? Requires baseline to establish anomaly

# Step 3: Postage stamp plots of CAE models at warming levels(?) and compared to CMIP6 models to show range with specific regions (**California**) shown
- min
- max
- mean
- median
- anomaly comparison between CAE models and CMIP6 multi-model ensemble mean?

In [ ]:
# similar to _GCM_PostageStamps_STATS in warming_levels